In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. 패키지

In [3]:
# python-dotenv, Langchain, Langchain-openai, Langchain-pinecone, Langchain-community, docx2txt
# Langchain-upstage, Langchain-ollama
%pip show pandas

Name: pandas
Version: 1.5.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: The Pandas Development Team
Author-email: pandas-dev@python.org
License: BSD-3-Clause
Location: c:\users\yisy0\anaconda3\envs\llm\lib\site-packages
Requires: numpy, python-dateutil, pytz
Required-by: seaborn
Note: you may need to restart the kernel to use updated packages.


# 2. 환경변수 설정

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
OPEBAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4092

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_REGIO = 'us-east-1'
PINECONE_INDEX_DEMENSION = 3072

# 3. 문서를 chunk로 분할하기

In [10]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('tax_docx/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 1500글자 기준으로 분리
    chunk_overlap=100,
    separators=['\n\n', '\n', ' ', '']
)
# documents = loader.load_and_split(text_splitter=text_splitter)
document = loader.load()
documents = text_splitter.split_documents(document)
print(f"총 {len(documents)}개 청크로 분리")

총 184개 청크로 분리


# 4. metadata 추가하기
- chunk 내용의 title, category, 조항

In [12]:
import re
def remove_special_chars(text:str) -> str:
    "특수문자 및 \n제거 (:, .는 그대로)"
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, . , : |-만 남기고 전부 제거
    cleanded = re.sub(r"[^가-힣a-zA-Z0-9\s\.,:]", "", text).strip()
    # 불필요한 중복 공백을 제거
    cleaned = re.sum(r"\s+", " ", cleanded)
    return cleaned
    
content = "**소득세 납세 의무** : \n\n이 법 또는 「범인세법」에 따라 소득에 대한 소득세를 부과"
remove_special_chars(content)

In [19]:
# 제목을 추출하는 함수
# 모델 설치: 터미널에서 ollama pull exaone3.5:2.4b
from langchain_ollama import ChatOllama

from langchain_ollama import ChatOllama

def extract_title_with_llm(content):
    """EXAONE 3.5를 사용하여 content의 제목을 반환"""
    title_extractor_llm = ChatOllama(
        model='exaone3.5:2.4b',
        temperature=0.1
    )

    prompt = f"""다음 소득세법 조문의 핵심 제목을 100토큰 이내로 추출해 주세요.
문장이 중간에 끊기지 않도록 하고, 제목만 출력해 주세요.
조문: {content}"""

    ai_message = title_extractor_llm.invoke(prompt)
    return ai_message.content.strip()

# 함수 실행
title = extract_title_with_llm(documents[0].page_content)
print(title)

**소득세법**

- **소득세 부과 원칙**: 소득 성격과 납세자 부담능력에 따른 적정 과세로 조세 형평 도모 및 재정 수입 확보
- **용어 정의**: 거주자, 비거주자, 내국법인, 외국법인, 사업자 구분


In [28]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }

In [29]:
def categorize_content(content, top_k=None):
    "content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 반환(top_k만큼)"
    pass

- 개선된 카테고리 분류방법 : 키워드의 중요도에 따라 가중치를 부여하고, 가중치가 높은 순으로 카테고리를 반환